# AWS RL Agent — SFT with LoRA on Qwen2.5-Coder-3B

**Pipeline**: HF SFT dataset → Optuna-tuned LoRA SFT → HF Hub adapter → GRPO (next phase)

| | |
|---|---|
| **Base model** | `Qwen/Qwen2.5-Coder-3B-Instruct` (4-bit via Unsloth) |
| **Dataset** | [`Sizzing/aws-rl-sft`](https://huggingface.co/datasets/Sizzing/aws-rl-sft) — 1500 train / 150 val |
| **Target GPU** | Kaggle dual-T4 (single T4 is enough with 4-bit) |
| **Expected runtime** | ~45 min end-to-end |
| **Logging** | Wandb `sizzing-sizzing/AWS-RL-SFT` |

## What this notebook does

1. **Pre-SFT baseline eval** — zero-shot metrics on the base model (the "before")
2. **Optuna search** — 6 trials on a 500-row subset to pick best LoRA hparams
3. **Final SFT** — full dataset with winning hparams, wandb-logged, checkpointed to disk
4. **Post-SFT eval** — same prompts, measure the delta
5. **Push adapter** — 60MB LoRA adapter to HF Hub (not the full 3B model)

## Why this stack

- **Unsloth** — ~2× training speed and half the VRAM via fused kernels (makes 3B fit on a single T4)
- **Optuna** — systematic hparam search instead of one-shot guessing; produces the parallel-coord plot judges love
- **TRL `SFTTrainer`** — handles the `messages` column and chat template automatically
- **Wandb** — per-trial + final run, all plots in one dashboard

## Expected deltas (from the pre-training eval)

| Metric | Before | Target after |
|---|--:|--:|
| `format%` | 85% | 100% |
| `exact-match%` | 41% | 75%+ |
| `operation%` | 63% | 90%+ |

## Checkpoint / resume

Final training saves every 50 steps to `/kaggle/working/final_sft/`. If the session dies, rerunning the final-training cell auto-detects the latest `checkpoint-*/` and resumes from it.

## 1. Install dependencies

Unsloth pulls in `trl`, `peft`, `accelerate`, `bitsandbytes`, and `transformers` at compatible versions. This takes ~3-4 min on first run.

In [ ]:
%%capture
# Unsloth ships platform-specific extras pinning xformers / triton / torch
# to versions that match the runtime. Pick the right one based on host.
import os as _os, sys as _sys
IS_KAGGLE = False
IS_COLAB = True

!pip install -q --upgrade pip

# if IS_KAGGLE:
#     !pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
# elif IS_COLAB:
#     !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# else:
#     !pip install -q unsloth

!pip install -q unsloth

!pip install -q --force-reinstall --no-deps "transformers>=4.50,<5.0"

!pip install -q --upgrade "trl<0.12.0" peft accelerate datasets huggingface_hub bitsandbytes
!pip install -q --upgrade optuna optuna-integration plotly kaleido wandb

## 2. Runtime sanity check

Confirm the GPU, pick the right output directory (Kaggle vs Colab vs local), and fail loudly if there's no GPU.

In [ ]:
import os, sys, json, time, gc, re
from pathlib import Path
import torch

# Reduce CUDA fragmentation on small GPUs (T4 has only 16 GB).
# Must be set BEFORE any CUDA allocation happens.
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')


if IS_KAGGLE:
    OUT_DIR = Path('/kaggle/working')
elif IS_COLAB:
    OUT_DIR = Path('/content/out')
else:
    OUT_DIR = Path('./out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'No GPU detected — enable GPU in runtime settings.'
gpu = torch.cuda.get_device_properties(0)

print(f'Environment  : {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Local"}')
print(f'Output dir   : {OUT_DIR}')
print(f'GPU          : {gpu.name}')
print(f'VRAM         : {gpu.total_memory / 1e9:.1f} GB')
print(f'GPU count    : {torch.cuda.device_count()}')
print(f'Torch        : {torch.__version__}')
print(f'CUDA         : {torch.version.cuda}')

# T4 (Turing) does not support bf16. We use fp16 throughout. Unsloth also
# handles this internally, but being explicit avoids silent conversions.
IS_T4 = 'T4' in gpu.name
USE_FP16 = IS_T4
USE_BF16 = not IS_T4
print(f'Precision    : {"fp16" if USE_FP16 else "bf16"}')

## 3. Config

Every knob lives in one dict. Fixed hyperparameters (batch size, epochs, target modules) stay here; Optuna tunes the ones below marked `(TUNED)`.

In [ ]:
CONFIG = {
    # --- Model ---
    # Unsloth's 4-bit pre-quantized version loads ~4× faster than bnb-on-the-fly
    'base_model': 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit',
    'max_seq_length': 512,          # our dataset p99 is 453; 512 covers everything
    'load_in_4bit': True,

    # --- Dataset ---
    'dataset_repo': 'Sizzing/aws-rl-sft',

    # --- Fixed SFT hyperparameters ---
    'per_device_train_batch_size': 4,    # bs=2 left GPU at 45%; bs=4 uses ~10/15 GB on T4
    'gradient_accumulation_steps': 4,    # effective batch = 16 (unchanged)
    'num_train_epochs': 2,               # full final run
    'optim': 'adamw_8bit',               # Unsloth-compatible, saves VRAM
    'max_grad_norm': 1.0,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.0,                 # LoRA is self-regularizing
    'seed': 42,
    'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],

    # --- Optuna search (TUNED) ---
    'n_trials': 6,
    'trial_max_train_rows': 500,         # subset for fast trials
    'trial_epochs': 1,                   # one pass is enough to rank hparams

    # --- Checkpointing ---
    'save_steps': 50,
    'save_total_limit': 3,               # keep last 3 checkpoints
    'logging_steps': 10,
    'eval_steps': 50,

    # --- Output ---
    'adapter_repo': 'Sizzing/aws-rl-sft-qwen25coder3b-adapter',
    'adapter_private': True,

    # --- Wandb ---
    'wandb_entity': 'sizzing-sizzing',
    'wandb_project': 'AWS-RL-SFT',
}

os.environ['WANDB_ENTITY'] = CONFIG['wandb_entity']
os.environ['WANDB_PROJECT'] = CONFIG['wandb_project']

CONFIG

## 4. Authenticate

Both `HF_TOKEN` and `WANDB_API_KEY` must be set.

**On Kaggle**: Notebook → Add-ons → Secrets → add both keys. The cell below picks them up automatically.
**On Colab**: `from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')`
**Locally**: export both as environment variables.

In [ ]:
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
elif IS_COLAB:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

assert 'HF_TOKEN' in os.environ and os.environ['HF_TOKEN'], 'HF_TOKEN missing'
assert 'WANDB_API_KEY' in os.environ and os.environ['WANDB_API_KEY'], 'WANDB_API_KEY missing'

from huggingface_hub import login as hf_login
import wandb
hf_login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
wandb.login(key=os.environ['WANDB_API_KEY'], verify=True)
print('OK: HF + Wandb authenticated')

## 5. Load dataset

The dataset is already published at `Sizzing/aws-rl-sft`. Each row has `messages` (chat format) plus metadata columns (`task_id`, `difficulty`, `source`, `step_idx`) for filtering.

In [ ]:
from datasets import load_dataset

ds = load_dataset(CONFIG['dataset_repo'], token=os.environ['HF_TOKEN'])
print(ds)

# Sanity: look at one row
row = ds['train'][0]
print(f'\ntask_id={row["task_id"]}  difficulty={row["difficulty"]}  source={row["source"]}')
for m in row['messages']:
    preview = m['content'][:140].replace('\n', ' ')
    print(f'  [{m["role"]:<9}] {preview}')

## 6. Eval harness — define

We'll use this function twice: once on the base model (baseline) and once after training (delta). Same prompts, same scoring — direct before/after comparison.

**Metrics** (match the ones from the model-selection eval):

- `format%` — output starts with `aws ` (no preamble)
- `format_after_extract%` — starts with `aws ` after stripping fences/prose
- `exact%` — extracted command matches canonical exactly
- `service%` — same AWS service (e.g., both `aws s3api`)
- `operation%` — same service AND operation (e.g., both `aws s3api create-bucket`)

In [ ]:
def extract_command(raw: str) -> str:
    """Strip fences/prose to find the first 'aws ...' line."""
    text = raw.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        text = '\n'.join(l for l in lines if not l.startswith('```')).strip()
    for line in text.split('\n'):
        line = line.strip()
        if line.startswith('aws '):
            return line
    return text

def score_row(completion: str, expected: str) -> dict:
    extracted = extract_command(completion)
    e_tokens = extracted.split()
    exp_tokens = expected.split()
    return {
        'format_ok': completion.strip().startswith('aws '),
        'format_after_extract': extracted.startswith('aws '),
        'exact': extracted == expected.strip(),
        'service': (len(e_tokens) >= 2 and len(exp_tokens) >= 2 and e_tokens[1:2] == exp_tokens[1:2]),
        'operation': (len(e_tokens) >= 3 and len(exp_tokens) >= 3 and e_tokens[2:3] == exp_tokens[2:3]),
    }

def curate_eval_set(dataset, max_per_combo: int = 2):
    seen = {}
    picks = []
    for r in dataset:
        key = (r['difficulty'], r['source'])
        seen[key] = seen.get(key, 0) + 1
        if seen[key] <= max_per_combo:
            picks.append(r)
    return picks

def eval_model(model, tokenizer, eval_set, max_new_tokens: int = 120) -> dict:
    results = []
    model.eval()
    for row in eval_set:
        msgs = row['messages'][:2]  # system + user only
        expected = row['messages'][2]['content']
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        t0 = time.time()
        with torch.inference_mode():
            out_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        dt = time.time() - t0
        completion = tokenizer.decode(out_ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        s = score_row(completion, expected)
        s.update({'latency': dt, 'len': len(completion), 'completion': completion, 'expected': expected})
        results.append(s)
    n = len(results)
    return {
        'format_pct':               sum(r['format_ok'] for r in results) / n,
        'format_after_extract_pct': sum(r['format_after_extract'] for r in results) / n,
        'exact_pct':                sum(r['exact'] for r in results) / n,
        'service_pct':              sum(r['service'] for r in results) / n,
        'operation_pct':            sum(r['operation'] for r in results) / n,
        'avg_latency':              sum(r['latency'] for r in results) / n,
        'avg_len':                  sum(r['len'] for r in results) / n,
        '_per_row':                 results,
    }

EVAL_SET = curate_eval_set(ds['validation'], max_per_combo=2)
combos = len(set((r['difficulty'], r['source']) for r in EVAL_SET))
print(f'Eval set: {len(EVAL_SET)} prompts across {combos} (tier, source) combos')

## 7. Pre-SFT baseline eval

Load Qwen2.5-Coder-3B in 4-bit and run the eval set. This is our **"before"** snapshot — we'll compare against it after training.

In [ ]:
from unsloth import FastLanguageModel

# Load base model (will cache on disk after first run)
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['base_model'],
    max_seq_length=CONFIG['max_seq_length'],
    load_in_4bit=CONFIG['load_in_4bit'],
    dtype=None,  # auto-select based on GPU
)
FastLanguageModel.for_inference(base_model)  # 2× faster generation via Unsloth kernels

print('Running pre-SFT baseline eval...')
baseline_metrics = eval_model(base_model, base_tokenizer, EVAL_SET)

print('\n=== PRE-SFT BASELINE ===')
for k, v in baseline_metrics.items():
    if k == '_per_row':
        continue
    if 'pct' in k:
        print(f'  {k:<30} {100*v:5.1f}%')
    else:
        print(f'  {k:<30} {v:.2f}')

# Save baseline to disk so we can compare later even if session restarts
with open(OUT_DIR / 'baseline_metrics.json', 'w') as f:
    json.dump({k: v for k, v in baseline_metrics.items() if k != '_per_row'}, f, indent=2)

# Free memory before Optuna starts reloading models per trial
del base_model, base_tokenizer
gc.collect(); torch.cuda.empty_cache()

## 8. Optuna — objective function

Each trial: load fresh base model → apply LoRA with trial's suggested params → train on 500-row subset for 1 epoch → return eval loss.

**Search space** (5 tunable knobs):

| Param | Range | Intuition |
|---|---|---|
| `lora_r` | {8, 16, 32} | adapter capacity |
| `lora_alpha_mul` | {1, 2, 4} | effective α = r × mul |
| `lora_dropout` | [0.0, 0.1] | regularization |
| `learning_rate` | [1e-4, 5e-4] log | the biggest lever |
| `warmup_ratio` | {0.03, 0.1} | stability at start |

In [ ]:
# --- Transformers 5.x compatibility shim (no restart needed) ---
# Unsloth 2026.x forwards `tokenizer=` through **kwargs to Trainer.__init__.
# Transformers 5.0+ removed that kwarg in favor of `processing_class=`.
# We patch Unsloth's captured reference so the rename happens at the boundary.
try:
    import transformers as _tf
    import unsloth.models._utils as _u
    _major = int(_tf.__version__.split('.')[0])
    if _major >= 5 and not getattr(_u, '_unsloth_tokenizer_shim', False):
        _orig_init = _u._original_trainer_init
        def _shimmed_init(self, *args, **kwargs):
            if 'tokenizer' in kwargs:
                kwargs['processing_class'] = kwargs.pop('tokenizer')
            return _orig_init(self, *args, **kwargs)
        _u._original_trainer_init = _shimmed_init
        _u._unsloth_tokenizer_shim = True
        print(f'Applied Transformers {_tf.__version__} shim: tokenizer= -> processing_class=')
    else:
        print(f'Transformers {_tf.__version__}: no shim needed')
except Exception as _e:
    print(f'Shim skipped: {_e}')

import optuna
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Prepare trial-sized splits once (not per trial)
TRIAL_TRAIN = ds['train'].shuffle(seed=CONFIG['seed']).select(range(CONFIG['trial_max_train_rows']))
TRIAL_VAL = ds['validation'].select(range(min(80, len(ds['validation']))))
print(f'Trial train: {len(TRIAL_TRAIN)}, trial val: {len(TRIAL_VAL)}')

def _render_messages(example, tokenizer):
    """Apply the chat template to turn messages into a single text string.
    TRL 0.12+ requires an explicit text column; Unsloth respects the
    tokenizer's built-in Qwen2.5 ChatML template.
    """
    return {'text': tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False,
    )}

def objective(trial: optuna.Trial) -> float:
    """One Optuna trial. Returns eval_loss (minimize)."""
    gc.collect(); torch.cuda.empty_cache()

    # --- Suggest hyperparameters ---
    lora_r          = trial.suggest_categorical('lora_r', [8, 16, 32])
    lora_alpha_mul  = trial.suggest_categorical('lora_alpha_mul', [1, 2, 4])
    lora_alpha      = lora_r * lora_alpha_mul
    lora_dropout    = trial.suggest_float('lora_dropout', 0.0, 0.1)
    learning_rate   = trial.suggest_float('learning_rate', 1e-4, 5e-4, log=True)
    warmup_ratio    = trial.suggest_categorical('warmup_ratio', [0.03, 0.1])

    # --- Load fresh model + tokenizer for this trial ---
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CONFIG['base_model'],
        max_seq_length=CONFIG['max_seq_length'],
        load_in_4bit=CONFIG['load_in_4bit'],
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=CONFIG['lora_target_modules'],
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=CONFIG['seed'],
    )

    # --- Render messages -> text using THIS trial's tokenizer ---
    train_ds = TRIAL_TRAIN.map(lambda ex: _render_messages(ex, tokenizer))
    val_ds   = TRIAL_VAL.map(lambda ex: _render_messages(ex, tokenizer))
    # Drop metadata and pre-tokenize, leaving only numeric columns.
    # Unsloth forces remove_unused_columns=False; if `text` survives into
    # the collator it raises 'too many dimensions str'. Pre-tokenizing and
    # skipping the trainer's internal dataset prep sidesteps both issues.
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c != 'text'])
    val_ds   = val_ds.remove_columns([c for c in val_ds.column_names if c != 'text'])
    def _tokenize(batch):
        return tokenizer(
            batch['text'], truncation=True,
            max_length=CONFIG['max_seq_length'], padding=False,
        )
    train_ds = train_ds.map(_tokenize, batched=True, remove_columns=['text'])
    val_ds   = val_ds.map(_tokenize, batched=True, remove_columns=['text'])

    trial_dir = OUT_DIR / f'optuna/trial-{trial.number}'
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=SFTConfig(
            output_dir=str(trial_dir),
            dataset_text_field='text',
            dataset_kwargs={'skip_prepare_dataset': True},  # we pre-tokenized above
            max_seq_length=CONFIG['max_seq_length'],
            per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
            gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
            num_train_epochs=CONFIG['trial_epochs'],
            learning_rate=learning_rate,
            warmup_ratio=warmup_ratio,
            lr_scheduler_type=CONFIG['lr_scheduler_type'],
            optim=CONFIG['optim'],
            weight_decay=CONFIG['weight_decay'],
            max_grad_norm=CONFIG['max_grad_norm'],
            fp16=USE_FP16, bf16=USE_BF16,
            logging_steps=25,
            eval_strategy='epoch',
            save_strategy='no',                    # trials don't save
            report_to='wandb',
            run_name=f'optuna-trial-{trial.number}',
            seed=CONFIG['seed'],
            dataset_num_proc=1,
        ),
    )
    # Completion-only loss: mask system + user tokens; train only on the
    # assistant's reply. These ChatML delimiters are Qwen2.5-specific.
    trainer = train_on_responses_only(
        trainer,
        instruction_part='<|im_start|>user\n',
        response_part='<|im_start|>assistant\n',
    )

    trainer.train()
    eval_result = trainer.evaluate()

    if wandb.run is not None:
        wandb.finish()

    loss = eval_result['eval_loss']
    print(f'  Trial {trial.number}: r={lora_r} alpha={lora_alpha} dropout={lora_dropout:.3f} '
          f'lr={learning_rate:.2e} warmup={warmup_ratio} -> eval_loss={loss:.4f}')

    del model, tokenizer, trainer, train_ds, val_ds
    gc.collect(); torch.cuda.empty_cache()
    return loss

## 9. Run the Optuna study

TPE sampler (Bayesian-ish) is more sample-efficient than random with only 6 trials.

In [ ]:
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=CONFIG['seed']),
    study_name='aws-rl-sft-lora-search',
)
study.optimize(objective, n_trials=CONFIG['n_trials'])

print('\n=== OPTUNA RESULTS ===')
print(f'Best eval_loss : {study.best_value:.4f}')
print(f'Best trial     : #{study.best_trial.number}')
print(f'Best params    :')
for k, v in study.best_params.items():
    print(f'  {k:<20} {v}')

# Persist the study so we can replot without rerunning
with open(OUT_DIR / 'optuna_study.json', 'w') as f:
    json.dump({
        'best_value': study.best_value,
        'best_params': study.best_params,
        'trials': [{'number': t.number, 'value': t.value, 'params': t.params, 'state': str(t.state)}
                   for t in study.trials],
    }, f, indent=2)

## 10. Optuna plots

Three views for the judges:

1. **Optimization history** — did later trials converge on lower loss?
2. **Parallel coordinates** — which hparam combinations trend toward low loss?
3. **Param importances** — which knob mattered most?

In [ ]:
from optuna.visualization import (
    plot_optimization_history, plot_parallel_coordinate, plot_param_importances,
)

plot_optimization_history(study).show()
plot_parallel_coordinate(study).show()
plot_param_importances(study).show()

## 11. Final SFT run — best hparams, full data, checkpointed

- Full dataset (1500 train / 150 val)
- 2 epochs (vs. 1 during search)
- Checkpoints saved every 50 steps; last 3 kept (`save_total_limit=3`)
- `load_best_model_at_end=True` — final model = lowest eval_loss checkpoint, not last
- **Resume-safe**: if the session dies, rerunning this cell picks up from the latest `checkpoint-*/`

In [ ]:
# Ensure the Transformers 5.x shim is in place (idempotent).
try:
    import transformers as _tf
    import unsloth.models._utils as _u
    if int(_tf.__version__.split('.')[0]) >= 5 and not getattr(_u, '_unsloth_tokenizer_shim', False):
        _orig_init = _u._original_trainer_init
        def _shimmed_init(self, *args, **kwargs):
            if 'tokenizer' in kwargs:
                kwargs['processing_class'] = kwargs.pop('tokenizer')
            return _orig_init(self, *args, **kwargs)
        _u._original_trainer_init = _shimmed_init
        _u._unsloth_tokenizer_shim = True
except Exception:
    pass

gc.collect(); torch.cuda.empty_cache()

best = study.best_params
final_r = best['lora_r']
final_alpha = final_r * best['lora_alpha_mul']

# --- Fresh model with winning hparams ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['base_model'],
    max_seq_length=CONFIG['max_seq_length'],
    load_in_4bit=CONFIG['load_in_4bit'],
)
model = FastLanguageModel.get_peft_model(
    model,
    r=final_r,
    lora_alpha=final_alpha,
    lora_dropout=best['lora_dropout'],
    target_modules=CONFIG['lora_target_modules'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=CONFIG['seed'],
)

# --- Render the full dataset with the chat template ---
full_train = ds['train'].map(lambda ex: _render_messages(ex, tokenizer))
full_val   = ds['validation'].map(lambda ex: _render_messages(ex, tokenizer))
# Drop metadata then pre-tokenize — leaves only numeric columns for the collator.
full_train = full_train.remove_columns([c for c in full_train.column_names if c != 'text'])
full_val   = full_val.remove_columns([c for c in full_val.column_names if c != 'text'])
def _tokenize_batch(batch):
    return tokenizer(
        batch['text'], truncation=True,
        max_length=CONFIG['max_seq_length'], padding=False,
    )
full_train = full_train.map(_tokenize_batch, batched=True, remove_columns=['text'])
full_val   = full_val.map(_tokenize_batch, batched=True, remove_columns=['text'])

# --- Auto-detect existing checkpoint to resume ---
ckpt_root = OUT_DIR / 'final_sft'
resume_ckpt = None
if ckpt_root.exists():
    existing = sorted(
        [d for d in ckpt_root.glob('checkpoint-*') if d.is_dir()],
        key=lambda d: int(d.name.split('-')[-1]),
    )
    if existing:
        resume_ckpt = str(existing[-1])
        print(f'Resuming from checkpoint: {resume_ckpt}')
    else:
        print('No checkpoint found — starting fresh')
else:
    print('No prior training dir — starting fresh')

final_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=full_train,
    eval_dataset=full_val,
    args=SFTConfig(
        output_dir=str(ckpt_root),
        dataset_text_field='text',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_seq_length=CONFIG['max_seq_length'],
        per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
        num_train_epochs=CONFIG['num_train_epochs'],
        learning_rate=best['learning_rate'],
        warmup_ratio=best['warmup_ratio'],
        lr_scheduler_type=CONFIG['lr_scheduler_type'],
        optim=CONFIG['optim'],
        weight_decay=CONFIG['weight_decay'],
        max_grad_norm=CONFIG['max_grad_norm'],
        fp16=USE_FP16, bf16=USE_BF16,
        eval_strategy='steps',
        eval_steps=CONFIG['eval_steps'],
        save_strategy='steps',
        save_steps=CONFIG['save_steps'],
        save_total_limit=CONFIG['save_total_limit'],
        logging_steps=CONFIG['logging_steps'],
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to='wandb',
        run_name='final-sft',
        seed=CONFIG['seed'],
        dataset_num_proc=1,
    ),
)
# Completion-only loss — same masking as Optuna trials
final_trainer = train_on_responses_only(
    final_trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

train_result = final_trainer.train(resume_from_checkpoint=resume_ckpt)
print(f'\nFinal train loss: {train_result.training_loss:.4f}')
final_eval = final_trainer.evaluate()
print(f'Final eval loss:  {final_eval["eval_loss"]:.4f}')

## 12. Post-SFT eval — the delta that matters

Same prompts as the baseline. This is the headline table for judges.

In [ ]:
FastLanguageModel.for_inference(model)

print('Running post-SFT eval on the same EVAL_SET...')
posttrain_metrics = eval_model(model, tokenizer, EVAL_SET)

print('\n=== BEFORE vs AFTER ===')
print(f'{"Metric":<30} {"Before":>10} {"After":>10} {"Delta":>12}')
print('-' * 64)
metric_keys = ['format_pct', 'format_after_extract_pct', 'exact_pct',
               'service_pct', 'operation_pct', 'avg_latency', 'avg_len']
for k in metric_keys:
    b = baseline_metrics[k]
    a = posttrain_metrics[k]
    if 'pct' in k:
        print(f'{k:<30} {100*b:9.1f}% {100*a:9.1f}% {100*(a-b):+11.1f}pt')
    else:
        print(f'{k:<30} {b:10.2f} {a:10.2f} {a-b:+12.2f}')

# Persist for the write-up
with open(OUT_DIR / 'delta_summary.json', 'w') as f:
    json.dump({
        'baseline': {k: baseline_metrics[k] for k in metric_keys},
        'posttrain': {k: posttrain_metrics[k] for k in metric_keys},
        'delta': {k: posttrain_metrics[k] - baseline_metrics[k] for k in metric_keys},
    }, f, indent=2)

# Log the headline deltas to wandb so they show up in the final run's summary panel
wandb.log({
    'delta/format_pct':    posttrain_metrics['format_pct']    - baseline_metrics['format_pct'],
    'delta/exact_pct':     posttrain_metrics['exact_pct']     - baseline_metrics['exact_pct'],
    'delta/operation_pct': posttrain_metrics['operation_pct'] - baseline_metrics['operation_pct'],
    'delta/service_pct':   posttrain_metrics['service_pct']   - baseline_metrics['service_pct'],
})

## 13. Inspect a few post-SFT generations

Qualitative sanity: does the trained model actually produce valid AWS commands?

In [ ]:
print('Post-SFT generations vs canonical:')
for r in posttrain_metrics['_per_row'][:6]:
    match = 'OK' if r['exact'] else ('~ ' if r['format_after_extract'] else 'X ')
    print(f'\n  [{match}] expected : {r["expected"][:100]!r}')
    print(f'      generated: {r["completion"].strip()[:100]!r}')

## 14. Push LoRA adapter to Hugging Face Hub

Just the adapter — ~60MB instead of the full 3B (~6GB). Consumers will apply it on top of `Qwen/Qwen2.5-Coder-3B-Instruct` at inference time.

In [ ]:
adapter_local = OUT_DIR / 'adapter'
model.save_pretrained(str(adapter_local))
tokenizer.save_pretrained(str(adapter_local))
print(f'Adapter saved locally: {adapter_local}')

model.push_to_hub(CONFIG['adapter_repo'], private=CONFIG['adapter_private'], token=os.environ['HF_TOKEN'])
tokenizer.push_to_hub(CONFIG['adapter_repo'], private=CONFIG['adapter_private'], token=os.environ['HF_TOKEN'])
print(f'OK: adapter pushed to https://huggingface.co/{CONFIG["adapter_repo"]}')

## 15. How to use this adapter at inference time

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Sizzing/aws-rl-sft-qwen25coder3b-adapter',  # adapter repo
    max_seq_length=512,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
# Unsloth auto-pulls the base model and attaches the LoRA
```

## 16. Next phase — GRPO

With the SFT'd adapter published, the GRPO phase builds on top:

1. Load base model + this adapter as the policy
2. Same adapter, frozen, as the KL reference (ensures GRPO doesn't drift)
3. Roll out episodes against the live `aws-rl-env` (MiniStack)
4. GRPO with online env reward + optional `<think>` reasoning (R1-Zero style)

The adapter we just trained serves as both the starting policy and the KL anchor — SFT locks format and basic competence, GRPO refines task correctness with real reward signal.